<a href="https://colab.research.google.com/github/OmegaS48/Internship_NITP/blob/main/Intern_Day14_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pickle

In [ ]:
from sklearn.svm import SVC

In [ ]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)
train=pd.concat([pd.read_csv('SDNFlow_Dataset_Normal.csv'),pd.read_csv('SDNFlow_Dataset_attack.csv')])
train=train.drop(columns=['ip_proto','src_port','dst_port','flow_ID','eth_type','ipv4_src','ipv4_dst','TnBPDstIP','TnPPDstIP','TnBPSrcIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP'])
train.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
train['Category']=le.fit_transform(train['Category'])
train['Category'].value_counts()

In [ ]:
Y=train['Category']
X=train

In [ ]:
X=X.drop(columns='Category')
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)
from xgboost import XGBClassifier


xgb=XGBClassifier()


estimators=[('xgb',xgb)]


from sklearn.pipeline import Pipeline


pipe=Pipeline(steps=estimators)


pipe

In [ ]:
pip install scikit-optimize


In [ ]:
from skopt import BayesSearchCV


from skopt.space import Real,Categorical,Integer


search_space = {
    'max_depth': Integer(3, 8),
    'learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'n_estimators': Integer(100, 500),
    'subsample': Real(0.6, 1.0),
    'colsample_bytree': Real(0.6, 1.0),
    'reg_alpha': Real(0.0, 5.0),
    'reg_lambda': Real(0.1, 5.0),
    'gamma': Real(0.0, 5.0)
}

opt=BayesSearchCV(pipe,search_space,n_iter=10,cv=3,scoring='roc_auc',random_state=8)


opt = BayesSearchCV(
    xgb,
    search_spaces=search_space,
    n_iter=10,
    cv=3
)

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy='median')

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

from imblearn.over_sampling import RandomOverSampler

ros = RandomOverSampler(random_state=42)

X_train, Y_train = ros.fit_resample(X_train, Y_train)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from xgboost import XGBClassifier

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class TabTransformer(nn.Module):
    def __init__(
        self,
        input_dim,
        num_classes,
        embed_dim=32,
        num_heads=4,
        num_layers=2,
        dropout=0.2
    ):
        super(TabTransformer, self).__init__()

        # Feature embedding
        self.embedding = nn.Linear(input_dim, embed_dim)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):

        # Embed features
        x = self.embedding(x)

        # Add sequence dimension
        x = x.unsqueeze(1)

        # Transformer
        x = self.transformer(x)

        # Pooling
        x = x.mean(dim=1)

        # Classification
        x = self.classifier(x)

        return x

In [ ]:
model = TabTransformer(
    input_dim=X_train.shape[1],
    num_classes=len(set(Y_train)),
    embed_dim=32,
    num_heads=4,
    num_layers=2
)


In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
from torch.utils.data import Dataset, DataLoader

In [ ]:
class TabDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X.to_numpy(), dtype=torch.float32)
        self.y = torch.tensor(y.to_numpy(), dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = TabDataset(X_train, Y_train)
test_dataset = TabDataset(X_test, Y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TabTransformer(
    input_dim=X_train.shape[1],
    num_classes=len(np.unique(Y))
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 20

for epoch in range(epochs):

    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_X)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")


In [ ]:
model.eval()

predictions = []
actuals = []

with torch.no_grad():

    for batch_X, batch_y in test_loader:

        batch_X = batch_X.to(device)

        outputs = model(batch_X)

        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())
        actuals.extend(batch_y.numpy())

In [ ]:
from sklearn.metrics import accuracy_score, classification_report,confusion_matrix

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
accuracy = accuracy_score(actuals, predictions)

print("\nAccuracy:", accuracy)

print("\nClassification Report:\n")
print(
    classification_report(
        actuals,
        predictions
    )
)

print("\nConfusion Matrix:\n")
print(
    confusion_matrix(
        actuals,
        predictions
    )
)

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [ ]:
models=[LogisticRegression(max_iter=4000),SVC(kernel='linear'),KNeighborsClassifier(),RandomForestClassifier(),HistGradientBoostingClassifier(),XGBClassifier()]

In [ ]:
def compare():
  for model in models:
    model.fit(X_train,Y_train)
    test_predict=model.predict(X_test)
    accuracy_score(Y_test,test_predict)
    print(model)
    print(accuracy_score(Y_test,test_predict))

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# fill missing values with column mean
imputer = SimpleImputer(strategy='median')

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

In [ ]:
compare()

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()
0.429413463558076
